# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset citation:** Mugotitsa, B., Maina, D., Owoko, H., Akinyi, L., Greenfield, J., Todd, J., Kavu, T. and Kiragga, A. 2026. A FAIR2 Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa. Frontiers.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Access as object, not as dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We demonstrate how to programmatically list the record sets, their fields, and their column `@id`s, referencing all entities by their `@id`.

In [ ]:
# List all available RecordSets by their @id
record_sets = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        record_sets.append(rs['@id'])
else:
    # Fallback: Try direct recordSet property
    if hasattr(metadata, 'recordSet'):
        if isinstance(metadata.recordSet, list):
            for rs in metadata.recordSet:
                if isinstance(rs, dict) and '@id' in rs:
                    record_sets.append(rs['@id'])
                elif isinstance(rs, str):
                    record_sets.append(rs)
        elif isinstance(metadata.recordSet, str):
            record_sets.append(metadata.recordSet)

print("RecordSets (@id):")
for rs_id in record_sets:
    print(f" - {rs_id}")

# For each record set, list fields and columns (@id)
for rs_id in record_sets:
    print(f"\nRecordSet: {rs_id}")
    # Attempting to extract fields using mlcroissant's API
    rs_obj = dataset.metadata.find_by_id(rs_id)
    # Fields may be in different attributes; try cr:field or fields
    fields = []
    if rs_obj is not None:
        if hasattr(rs_obj, 'fields'):
            for f in rs_obj.fields:
                fid = f['@id'] if isinstance(f, dict) and '@id' in f else f if isinstance(f, str) else None
                if fid:
                    fields.append(fid)
        if hasattr(rs_obj, 'field'):
            for f in rs_obj.field:
                fid = f['@id'] if isinstance(f, dict) and '@id' in f else f if isinstance(f, str) else None
                if fid:
                    fields.append(fid)
    print("Fields (@id):")
    for fid in fields:
        print(f"   - {fid}")
    # For columns, search via mlcroissant API or via fields
    for field_id in fields:
        field_obj = dataset.metadata.find_by_id(field_id)
        print(f"     Field {field_id}: columns (@id):")
        columns = []
        if field_obj is not None:
            if hasattr(field_obj, 'columns'):
                for col in field_obj.columns:
                    col_id = col['@id'] if isinstance(col, dict) and '@id' in col else col if isinstance(col, str) else None
                    if col_id:
                        columns.append(col_id)
            if hasattr(field_obj, 'column'):
                for col in field_obj.column:
                    col_id = col['@id'] if isinstance(col, dict) and '@id' in col else col if isinstance(col, str) else None
                    if col_id:
                        columns.append(col_id)
        for col_id in columns:
            print(f"        - {col_id}")

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this dataset, all entities are referenced by their `@id` (record sets, fields, columns, etc.).

In [ ]:
# Use discovered record_sets
# In this example, we'll extract all available recordSets; modify for your use case.
dataframes = {}

for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from RecordSet {rs_id}")
    except Exception as e:
        print(f"Error loading records from {rs_id}: {e}")

# Show columns for each record set
for rs_id, df in dataframes.items():
    print(f"\nColumns for RecordSet {rs_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations may include removing outliers, transforming data distributions, or grouping data by key attributes. All field references use the field or column `@id`.

In [ ]:
# Select a record set and numeric field for analysis
# For example, suppose we have a RecordSet @id and a GAD-7 score column @id discovered previously
example_record_set_id = record_sets[0] if record_sets else None
# You may need to adjust the column @id manually if overview does not list them
example_numeric_field_id = None
if example_record_set_id:
    df = dataframes[example_record_set_id]
    # Try to pick a numeric column as example (e.g. 'gad_7_score' or similar)
    for col in df.columns:
        # Attempt to select GAD-7, PHQ-9, PSQ scores if available
        if 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower():
            example_numeric_field_id = col
            break
    if example_numeric_field_id:
        print(f"Using numeric field @id: {example_numeric_field_id}")

    threshold = 10
    # Filter out missing values and apply threshold
    filtered_df = df[df[example_numeric_field_id].apply(lambda x: pd.to_numeric(x, errors='coerce')).fillna(0) > threshold]
    print(f"Filtered records with {example_numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{example_numeric_field_id}_normalized"] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
    print(f"Normalized {example_numeric_field_id} for filtered records:")
    print(filtered_df[[example_numeric_field_id, f"{example_numeric_field_id}_normalized"]].head())

    # Group by a demographic field, e.g., 'gender' or 'age_group', referenced by @id
    example_group_field_id = None
    for col in df.columns:
        if col.lower() in ['gender', 'sex', 'age_group', 'marital_status', 'village']:
            example_group_field_id = col
            break
    if example_group_field_id:
        grouped_df = filtered_df.groupby(example_group_field_id)[example_numeric_field_id].mean()
        print(f"Grouped data by {example_group_field_id} (@id):")
        print(grouped_df.head())
else:
    print("No record sets or numeric fields found; please revise field selection.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we demonstrate a histogram of a numeric score, and a barplot by group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and example_numeric_field_id:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[example_numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {example_numeric_field_id} (@id)")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Barplot for grouped means if group field exists
    if example_group_field_id:
        plt.figure(figsize=(8, 4))
        grouped_means = df.groupby(example_group_field_id)[example_numeric_field_id].mean()
        grouped_means = grouped_means.dropna()
        grouped_means.plot(kind='bar')
        plt.title(f"Mean {example_numeric_field_id} by {example_group_field_id} (@id)")
        plt.xlabel(example_group_field_id)
        plt.ylabel(f"Mean {example_numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset contains rich demographic and mental health symptom data.
- Using `mlcroissant`, we loaded and explored multiple record sets and fields, referencing all entities by their `@id`.
- Visualizations reveal patterns in survey scores (e.g., GAD-7) and demographic groupings.
- This notebook can be extended for detailed statistical analysis or modeling using referenced fields and columns.

For full documentation, see [mlcroissant GitHub](https://github.com/mlcommons/croissant) and [dataset metadata](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).